In [1]:
import pandas as pd
import numpy as np

OPERATIONAL_PATH = "train_operational_readouts.csv"
TTE_PATH = "train_tte.csv"

In [2]:
tte = pd.read_csv(TTE_PATH)

print("TTE shape:", tte.shape)
print("\nTTE columns:")
print(tte.columns.tolist())

print("\nUnique vehicles in TTE:")
print(tte["vehicle_id"].nunique())

print("\nin_study_repair counts:")
print(tte["in_study_repair"].value_counts(dropna=False))

print("\nDuplicate TTE rows:")
print(tte.duplicated().sum())

print("\nDuplicate vehicle IDs in TTE:")
print(tte["vehicle_id"].duplicated().sum())

TTE shape: (23550, 3)

TTE columns:
['vehicle_id', 'length_of_study_time_step', 'in_study_repair']

Unique vehicles in TTE:
23550

in_study_repair counts:
in_study_repair
0    21278
1     2272
Name: count, dtype: int64

Duplicate TTE rows:
0

Duplicate vehicle IDs in TTE:
0


In [3]:
operational = pd.read_csv(
    OPERATIONAL_PATH,
    low_memory=False
)

print("Operational shape:", operational.shape)

print("\nFirst 5 rows:")
display(operational.head())

print("\nColumn count:")
print(len(operational.columns))

print("\nUnique vehicles:")
print(operational["vehicle_id"].nunique())

print("\nDuplicate rows:")
print(operational.duplicated().sum())

Operational shape: (1122452, 107)

First 5 rows:


,vehicle_id,time_step,171_0,666_0,427_0,837_0,167_0,167_1,167_2,167_3,...,397_26,397_27,397_28,397_29,397_30,397_31,397_32,397_33,397_34,397_35
0,0,11.2,167985.0,10787.0,7413813.0,2296.0,4110.0,1296420.0,1628265.0,630345.0,...,95728.0,15609.0,1984.0,8.0,784.0,150228.0,261904.0,93172.0,17874.0,452.0
1,0,11.4,167985.0,10787.0,7413813.0,2296.0,4111.0,1302855.0,1628265.0,630345.0,...,95729.0,15610.0,1984.0,8.0,784.0,150228.0,261905.0,93172.0,17874.0,452.0
2,0,19.6,331635.0,14525.0,13683604.0,2600.0,NaN,NaN,NaN,NaN,...,142900.0,19263.0,2441.0,12.0,1420.0,204832.0,313485.0,106464.0,19306.0,452.0
3,0,20.2,354975.0,15015.0,14540449.0,2616.0,NaN,NaN,NaN,NaN,...,150565.0,19832.0,2522.0,12.0,1444.0,211688.0,318901.0,107745.0,19406.0,453.0
4,0,21.0,365550.0,15295.0,14966985.0,2720.0,NaN,NaN,NaN,NaN,...,155913.0,20573.0,2562.0,12.0,1445.0,213956.0,323997.0,109514.0,19535.0,454.0



Column count:
107

Unique vehicles:
23550

Duplicate rows:
0


In [4]:
print("===== FULL OPERATIONAL DATASET =====")

print("Rows:", len(operational))
print("Columns:", len(operational.columns))
print("Unique vehicles:", operational["vehicle_id"].nunique())

print(
    "time_step range:",
    operational["time_step"].min(),
    "to",
    operational["time_step"].max()
)

print("Duplicate rows:", operational.duplicated().sum())

===== FULL OPERATIONAL DATASET =====
Rows: 1122452
Columns: 107
Unique vehicles: 23550
time_step range: 0.0 to 507.4
Duplicate rows: 0


In [5]:
tte["in_study_repair"] = pd.to_numeric(
    tte["in_study_repair"],
    errors="coerce"
)

event_vehicles = tte.loc[
    tte["in_study_repair"] == 1,
    "vehicle_id"
].nunique()

censored_vehicles = tte.loc[
    tte["in_study_repair"] == 0,
    "vehicle_id"
].nunique()

total_tte_vehicles = tte["vehicle_id"].nunique()

print("Total vehicles:", total_tte_vehicles)
print("Event/repair vehicles:", event_vehicles)
print("Censored vehicles:", censored_vehicles)
print("Check:", event_vehicles + censored_vehicles)

Total vehicles: 23550
Event/repair vehicles: 2272
Censored vehicles: 21278
Check: 23550


In [6]:
merged = operational.merge(
    tte,
    on="vehicle_id",
    how="left",
    validate="many_to_one"
)

print("Operational rows:", len(operational))
print("Merged rows:", len(merged))

print("\nRows changed after merge:")
print(len(merged) - len(operational))

print("\nVehicles missing TTE information:")
print(
    merged["length_of_study_time_step"].isna()
    .groupby(merged["vehicle_id"])
    .any()
    .sum()
)

Operational rows: 1122452
Merged rows: 1122452

Rows changed after merge:
0

Vehicles missing TTE information:
0


In [7]:
event_mask = merged["in_study_repair"] == 1
censored_mask = merged["in_study_repair"] == 0

event_rows = event_mask.sum()
censored_rows = censored_mask.sum()

print("Total operational observations:", len(merged))
print("Observations from event vehicles:", event_rows)
print("Observations from censored vehicles:", censored_rows)

print(
    "Check:",
    event_rows + censored_rows
)

Total operational observations: 1122452
Observations from event vehicles: 105852
Observations from censored vehicles: 1016600
Check: 1122452


In [8]:
merged["RUL"] = (
    merged["length_of_study_time_step"]
    - merged["time_step"]
)

event_data = merged[
    merged["in_study_repair"] == 1
].copy()

print("Event observations:", len(event_data))

print("\nRUL statistics:")
print(event_data["RUL"].describe())

print("\nNegative RUL values:")
print((event_data["RUL"] < 0).sum())

print("\nZero RUL values:")
print((event_data["RUL"] == 0).sum())

print("\nMissing RUL values:")
print(event_data["RUL"].isna().sum())

Event observations: 105852

RUL statistics:
count    105852.000000
mean        118.131351
std          84.351319
min           0.200000
25%          49.400000
50%         103.600000
75%         172.200000
max         482.000000
Name: RUL, dtype: float64

Negative RUL values:
0

Zero RUL values:
0

Missing RUL values:
0


In [9]:
missing_summary = pd.DataFrame({
    "missing_count": operational.isna().sum(),
    "missing_percentage": (
        operational.isna().mean() * 100
    )
})

missing_summary = (
    missing_summary[
        missing_summary["missing_count"] > 0
    ]
    .sort_values("missing_percentage", ascending=False)
)

print("Columns containing missing values:")
display(missing_summary.head(20))

Columns containing missing values:


,missing_count,missing_percentage
291_8,9628,0.857765
291_6,9628,0.857765
291_0,9628,0.857765
291_1,9628,0.857765
291_2,9628,0.857765
291_3,9628,0.857765
291_4,9628,0.857765
291_10,9628,0.857765
291_9,9628,0.857765
291_5,9628,0.857765


In [10]:
exclude_columns = [
    "vehicle_id",
    "time_step",
    "length_of_study_time_step",
    "in_study_repair",
    "RUL"
]

sensor_features = [
    col
    for col in merged.columns
    if col not in exclude_columns
    and pd.api.types.is_numeric_dtype(merged[col])
]

print("Number of selected numeric input features:")
print(len(sensor_features))

print("\nFeature names:")
print(sensor_features)

Number of selected numeric input features:
105

Feature names:
['171_0', '666_0', '427_0', '837_0', '167_0', '167_1', '167_2', '167_3', '167_4', '167_5', '167_6', '167_7', '167_8', '167_9', '309_0', '272_0', '272_1', '272_2', '272_3', '272_4', '272_5', '272_6', '272_7', '272_8', '272_9', '835_0', '370_0', '291_0', '291_1', '291_2', '291_3', '291_4', '291_5', '291_6', '291_7', '291_8', '291_9', '291_10', '158_0', '158_1', '158_2', '158_3', '158_4', '158_5', '158_6', '158_7', '158_8', '158_9', '100_0', '459_0', '459_1', '459_2', '459_3', '459_4', '459_5', '459_6', '459_7', '459_8', '459_9', '459_10', '459_11', '459_12', '459_13', '459_14', '459_15', '459_16', '459_17', '459_18', '459_19', '397_0', '397_1', '397_2', '397_3', '397_4', '397_5', '397_6', '397_7', '397_8', '397_9', '397_10', '397_11', '397_12', '397_13', '397_14', '397_15', '397_16', '397_17', '397_18', '397_19', '397_20', '397_21', '397_22', '397_23', '397_24', '397_25', '397_26', '397_27', '397_28', '397_29', '397_30', '397

In [12]:
total_missing_cells = operational.isna().sum().sum()

print(
    "Total missing cells in operational dataset:",
    total_missing_cells
)

Total missing cells in operational dataset: 354634


In [13]:
dataset_report = {
    "Operational observations": len(operational),
    "Operational columns": len(operational.columns),
    "Unique vehicles": operational["vehicle_id"].nunique(),
    "TTE vehicles": tte["vehicle_id"].nunique(),
    "Event vehicles": event_vehicles,
    "Censored vehicles": censored_vehicles,
    "Merged observations": len(merged),
    "Duplicate operational rows": operational.duplicated().sum(),
    "Minimum time_step": operational["time_step"].min(),
    "Maximum time_step": operational["time_step"].max(),
    "Event observations": event_rows,
    "Censored observations": censored_rows,
    "RUL observations": len(event_data),
    "Negative RUL": (event_data["RUL"] < 0).sum(),
    "Missing RUL": event_data["RUL"].isna().sum(),
    "Selected numeric features": len(sensor_features),
    "Total missing cells": total_missing_cells
}

report_df = pd.DataFrame(
    dataset_report.items(),
    columns=["Statistic", "Value"]
)

display(report_df)

,Statistic,Value
0,Operational observations,1122452.0
1,Operational columns,107.0
2,Unique vehicles,23550.0
3,TTE vehicles,23550.0
4,Event vehicles,2272.0
5,Censored vehicles,21278.0
6,Merged observations,1122452.0
7,Duplicate operational rows,0.0
8,Minimum time_step,0.0
9,Maximum time_step,507.4


In [14]:
report_df.to_csv(
    "full_dataset_statistics.csv",
    index=False
)

print("Saved: full_dataset_statistics.csv")

Saved: full_dataset_statistics.csv
